In [ ]:
%pip install pandas google-cloud-bigquery pyarrow

In [2]:
import pandas as pd
import os

from google.cloud import bigquery
from google.oauth2 import service_account

c:\Users\jainp\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [11]:
project_id = "casper-code-challenge"
dataset_id = "casper_data_raw"

credentials_path = (
    r"C:\Users\jainp\Downloads\casper-code-challenge-b2a02376a078.json"
)

data_dir = r"C:\Users\jainp\Desktop\My Repository\dbt_casper\raw_data"

In [5]:
credentials = service_account.Credentials.from_service_account_file(
    credentials_path
)

client = bigquery.Client(
    project=project_id,
    credentials=credentials,
    location="EU"
)

In [ ]:
# test the connection by running a simple query
query = "SELECT 1 AS test"

In [9]:
result = client.query(query).result()

for row in result:
    print(row.test)

1


In [13]:
csv_files = [
    file for file in os.listdir(data_dir)
    if file.lower().endswith(".csv")
]

csv_files

['exercises.csv', 'patients.csv', 'steps.csv']

In [14]:
for file in csv_files:
    file_path = os.path.join(data_dir, file)
    # read CSV file into a pandas dataframe
    df = pd.read_csv(file_path)

    # Use CSV filename as BigQuery table name
    table_name = os.path.splitext(file)[0]

    table_id = f"{project_id}.{dataset_id}.{table_name}"

    # Load dataframe into BigQuery
    job_config = bigquery.LoadJobConfig(
        autodetect=True,
        #replace existing table data rather than appending
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE 
    )

    load_job = client.load_table_from_dataframe(
        df,
        table_id,
        job_config=job_config
    )

    load_job.result()

    print(f"loaded {file} : {table_id}")

c:\Users\jainp\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


loaded exercises.csv : casper-code-challenge.casper_data_raw.exercises


c:\Users\jainp\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


loaded patients.csv : casper-code-challenge.casper_data_raw.patients


c:\Users\jainp\AppData\Local\Programs\Python\Python310\lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


loaded steps.csv : casper-code-challenge.casper_data_raw.steps
